# 06 — Confidence Gate

## Purpose
Scores each extracted document segment and routes it to either
`AUTO_APPROVE` or `HUMAN_REVIEW` based on a composite confidence score
and a set of explicit flag rules. Also catches documents that failed
earlier pipeline stages or were classified as unknown and routes them
directly to the review queue with a reason flag.

## What this notebook does
Computes a composite score for each segment from two signals — field
completeness (proportion of mandatory fields successfully extracted) and
extract certainty (average confidence across non-missing mandatory fields).
Signal weights are loaded from `DOC_TYPE_CONFIG` per doc type, defaulting
to 50/50 if not configured.

A segment is routed to `HUMAN_REVIEW` if any of the following flags fire,
regardless of composite score:

- `MISSING_MANDATORY_FIELDS` - one or more mandatory fields returned null
- `LOW_COMPOSITE_SCORE` - composite score below `AUTO_APPROVE_THRESHOLD`
- `LOW_CLASSIFY_CONFIDENCE` - classify confidence below
  `CLASSIFY_CONF_THRESHOLD`, meaning the doc type or page boundaries
  may be wrong
- `UNKNOWN_DOC_TYPE` - classifier could not identify the document type;
  no extraction was attempted
- `PIPELINE_ERROR` - document failed at an earlier stage (parse, classify,
  or extract); root cause investigation required

Only segments with no flags and a composite score at or above threshold
are routed to `AUTO_APPROVE`. Flag reasons are stored as a JSON array on
both `DOCUMENTS_SCORED` and `REVIEW_QUEUE` so the review portal can display
exactly why a document was flagged.

Status is updated in `DOCUMENTS_INGESTED` to `AUTO_APPROVED` or
`HUMAN_REVIEW`.

## Outputs
| Table | What is written |
|---|---|
| `PROCESSING.DOCUMENTS_SCORED` | One row per segment - composite score, signal breakdown, gate result, flag reasons |
| `PROCESSING.REVIEW_QUEUE` | One row per flagged segment - doc type, score, flag reasons, notes |
| `INGEST.DOCUMENTS_INGESTED` | STATUS updated to `AUTO_APPROVED` or `HUMAN_REVIEW` |

## Key design decisions
- **Any missing mandatory field forces review** - regardless of composite
  score; a document missing a required field cannot be auto-approved even
  if all other fields extracted with high confidence
- **Classify confidence as override signal not composite input** - low
  classify confidence flags the document for review without affecting the
  extraction score, since the two are independent failure modes
- **Pipeline errors caught here** - documents that failed parse, classify,
  or extract are routed to the review queue here rather than silently
  dropped, giving reviewers visibility into all documents that need
  attention in one place
- **Flag reasons stored as JSON array** - each flag is a descriptive string
  (e.g. `MISSING_MANDATORY_FIELDS:2_of_4`) rather than a boolean column,
  making it easy to add new flag types without schema changes and allowing
  the review portal to render a human-readable explanation per document

In [ ]:
import json
import pandas as pd
from snowflake.snowpark.context import get_active_session

DB                = 'PERMAFROST_POC'
PROCESSING_SCHEMA = 'PROCESSING'
INGEST_SCHEMA     = 'INGEST'
CONFIG_SCHEMA     = 'CONFIG'
AUDIT_SCHEMA      = 'AUDIT'

EXTRACTION_MODEL          = 'claude-sonnet-4-6'
AUTO_APPROVE_THRESHOLD    = 0.80
CLASSIFY_CONF_THRESHOLD   = 0.70   # below threshold - LOW_CLASSIFY_CONFIDENCE flag

def info(msg):    print(f"INFO:    {msg}")
def warning(msg): print(f"WARNING: {msg}")
def error(msg):   print(f"ERROR:   {msg}")

s = get_active_session()

In [ ]:
#Load confidence weights from DOC_TYPE_CONFIG
DEFAULT_WEIGHTS = {
    'field_completeness': 0.50,
    'extract_certainty':  0.50,
}

weights_cache = {}

doc_type_rows = s.sql(f"""
    SELECT DOC_TYPE, CONFIDENCE_WEIGHTS
    FROM {DB}.{CONFIG_SCHEMA}.DOC_TYPE_CONFIG
    WHERE IS_ACTIVE = TRUE
""").collect()

for row in doc_type_rows:
    doc_type = row['DOC_TYPE']
    raw      = row['CONFIDENCE_WEIGHTS']
    if raw:
        weights = json.loads(raw) if isinstance(raw, str) else raw
    else:
        weights = DEFAULT_WEIGHTS
    weights_cache[doc_type] = weights

info(f"Weights loaded for {len(weights_cache)} doc type(s)")

In [ ]:
# Pull all documents that need scoring
#  - Successfully classified and extracted documents
#  - Documents with pipeline errors (PARSE_ERROR, CLASSIFY_ERROR etc.)
#  - Documents classified as 'unknown'

# Successfully classified + extracted
extracted_docs = s.sql(f"""
    SELECT
        c.CHILD_DOC_ID,
        c.DOC_ID,
        c.DOC_TYPE,
        c.CONFIDENCE                                      AS CLASSIFY_CONFIDENCE,
        i.STATUS                                          AS INGEST_STATUS,
        -- Field completeness: mandatory fields present / total mandatory
        COUNT(CASE WHEN f.IS_MANDATORY = TRUE THEN 1 END)
            / NULLIF(SUM(CASE WHEN f.IS_MANDATORY = TRUE THEN 1 END), 0)
            * SUM(CASE WHEN f.IS_MANDATORY = TRUE
                       AND f.IS_MISSING   = FALSE THEN 1 ELSE 0 END)
            / NULLIF(COUNT(CASE WHEN f.IS_MANDATORY = TRUE THEN 1 END), 0)
                                                          AS FIELD_COMPLETENESS_RAW,
        -- Extract certainty: avg confidence on non-missing mandatory fields
        AVG(CASE WHEN f.IS_MANDATORY = TRUE
                  AND f.IS_MISSING   = FALSE
             THEN f.FIELD_CONFIDENCE END)                 AS EXTRACT_CERTAINTY_RAW,
        -- Missing mandatory count
        SUM(CASE WHEN f.IS_MANDATORY = TRUE
                  AND f.IS_MISSING   = TRUE THEN 1 ELSE 0 END)
                                                          AS MISSING_MANDATORY_COUNT,
        -- Total mandatory fields
        COUNT(CASE WHEN f.IS_MANDATORY = TRUE THEN 1 END) AS TOTAL_MANDATORY
    FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_CLASSIFIED c
    JOIN {DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED i
        ON c.DOC_ID = i.DOC_ID
    JOIN {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_EXTRACTED_FLAT f
        ON  c.CHILD_DOC_ID    = f.CHILD_DOC_ID
        AND f.EXTRACTION_MODEL = '{EXTRACTION_MODEL}'
    LEFT JOIN {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_SCORED sc
        ON  c.CHILD_DOC_ID    = sc.CHILD_DOC_ID
        AND sc.EXTRACTION_MODEL = '{EXTRACTION_MODEL}'
    WHERE sc.CHILD_DOC_ID IS NULL   
    GROUP BY
        c.CHILD_DOC_ID, c.DOC_ID,
        c.DOC_TYPE, c.CONFIDENCE, i.STATUS
""").collect()

# Documents with pipeline errors
error_docs = s.sql(f"""
    SELECT
        DOC_ID          AS DOC_ID,
        STATUS          AS INGEST_STATUS,
        ORIGINAL_FILENAME
    FROM {DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED
    WHERE STATUS IN ('PARSE_ERROR', 'CLASSIFY_ERROR', 'EXTRACT_ERROR')
""").collect()

info(f"Extracted docs to score  : {len(extracted_docs)}")
info(f"Error docs to route      : {len(error_docs)}")


# Documents classified as unknown
unknown_docs = s.sql(f"""
    SELECT
        c.CHILD_DOC_ID,
        c.DOC_ID,
        c.DOC_TYPE,
        c.CONFIDENCE    AS CLASSIFY_CONFIDENCE,
        i.ORIGINAL_FILENAME
    FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_CLASSIFIED c
    JOIN {DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED i
        ON c.DOC_ID = i.DOC_ID
    LEFT JOIN {DB}.{PROCESSING_SCHEMA}.REVIEW_QUEUE r
        ON c.CHILD_DOC_ID = r.CHILD_DOC_ID
    WHERE c.DOC_TYPE    = 'unknown'
      AND r.CHILD_DOC_ID IS NULL   -- not already in review queue
""").collect()

info(f"Unknown doc type to route: {len(unknown_docs)}")

In [ ]:

scored_rows      = []
review_rows      = []
auto_approve_ids = []

# Score extracted documents
for row in extracted_docs:
    child_doc_id          = row['CHILD_DOC_ID']
    doc_type              = row['DOC_TYPE']
    classify_confidence   = row['CLASSIFY_CONFIDENCE'] or 0.0
    missing_mandatory     = row['MISSING_MANDATORY_COUNT'] or 0
    total_mandatory       = row['TOTAL_MANDATORY'] or 0
    ingest_status         = row['INGEST_STATUS']

    # Field completeness — ratio of mandatory fields present
    if total_mandatory > 0:
        field_completeness = (total_mandatory - missing_mandatory) / total_mandatory
    else:
        field_completeness = 1.0

    # Extract certainty — avg confidence on non-missing mandatory fields
    extract_certainty = row['EXTRACT_CERTAINTY_RAW'] or 0.0

    # Weights from config
    weights = weights_cache.get(doc_type, DEFAULT_WEIGHTS)
    w_comp  = weights.get('field_completeness', 0.50)
    w_cert  = weights.get('extract_certainty',  0.50)

    # Composite score
    composite = (field_completeness * w_comp) + (extract_certainty * w_cert)

    # Flag reasons 
    flags = []

    if doc_type == 'unknown':
        flags.append('UNKNOWN_DOC_TYPE')

    if classify_confidence < CLASSIFY_CONF_THRESHOLD:
        flags.append(f'LOW_CLASSIFY_CONFIDENCE:{round(classify_confidence, 3)}')

    if missing_mandatory > 0:
        flags.append(f'MISSING_MANDATORY_FIELDS:{missing_mandatory}_of_{total_mandatory}')

    if composite < AUTO_APPROVE_THRESHOLD:
        flags.append(f'LOW_COMPOSITE_SCORE:{round(composite, 3)}')

    # Gate decision
    # Route to HUMAN_REVIEW if ANY flag exists
    # Auto-approve only when completely clean
    gate_result = 'HUMAN_REVIEW' if flags else 'AUTO_APPROVE'

    scored_rows.append({
        'CHILD_DOC_ID':        child_doc_id,
        'EXTRACTION_MODEL':    EXTRACTION_MODEL,
        'CLASSIFY_CONFIDENCE': classify_confidence,
        'FIELD_COMPLETENESS':  round(field_completeness, 4),
        'EXTRACT_CERTAINTY':   round(extract_certainty, 4),
        'COMPOSITE_SCORE':     round(composite, 4),
        'GATE_RESULT':         gate_result,
        'FLAG_REASONS':        json.dumps(flags),
    })

    if gate_result == 'HUMAN_REVIEW':
        review_rows.append({
            'CHILD_DOC_ID':  child_doc_id,
            'DOC_TYPE':      doc_type,
            'COMPOSITE_SCORE': round(composite, 4),
            'FLAG_REASONS':  json.dumps(flags),
            'NOTES':         f"Routed by confidence gate. Flags: {', '.join(flags)}",
            'STATUS':        'PENDING',
        })
    else:
        auto_approve_ids.append(child_doc_id)

    # Log per document
    flag_str = ', '.join(flags) if flags else '✓ clean'
    info(f"  [{gate_result}] {child_doc_id} ({doc_type}) "
         f"— composite: {composite:.3f} | {flag_str}")
         
# Route unknown doc type documents
for row in unknown_docs:
    child_doc_id        = row['CHILD_DOC_ID']
    classify_confidence = row['CLASSIFY_CONFIDENCE'] or 0.0
    filename            = row['ORIGINAL_FILENAME']

    flags = ['UNKNOWN_DOC_TYPE']
    if classify_confidence < CLASSIFY_CONF_THRESHOLD:
        flags.append(f'LOW_CLASSIFY_CONFIDENCE:{round(classify_confidence, 3)}')

    review_rows.append({
        'CHILD_DOC_ID':   child_doc_id,
        'DOC_TYPE':       'unknown',
        'COMPOSITE_SCORE': None,
        'FLAG_REASONS':   json.dumps(flags),
        'NOTES':          f"Document type could not be identified for "
                          f"'{filename}'. Manual classification required.",
        'STATUS':         'PENDING',
    })
    warning(f"  [UNKNOWN] {filename} — classify confidence: {classify_confidence:.3f}")

# Route pipeline error documents
for row in error_docs:
    parent_doc_id = row['DOC_ID']
    status        = row['INGEST_STATUS']
    filename      = row['ORIGINAL_FILENAME']

    flag = {
        'PARSE_ERROR':    'PIPELINE_ERROR:PARSE_FAILED',
        'CLASSIFY_ERROR': 'PIPELINE_ERROR:CLASSIFY_FAILED',
        'EXTRACT_ERROR':  'PIPELINE_ERROR:EXTRACT_FAILED',
    }.get(status, f'PIPELINE_ERROR:{status}')

    review_rows.append({
        'CHILD_DOC_ID':   parent_doc_id,
        'DOC_TYPE':       None,
        'COMPOSITE_SCORE': None,
        'FLAG_REASONS':   json.dumps([flag]),
        'NOTES':          f"Pipeline error on '{filename}' — status: {status}. "
                          f"Manual investigation required.",
        'STATUS':         'PENDING',
    })
    warning(f"  [PIPELINE ERROR] {filename} — {flag}")


# Write DOCUMENTS_SCORED
if scored_rows:
    s.write_pandas(
        pd.DataFrame(scored_rows),
        table_name='DOCUMENTS_SCORED',
        database=DB, schema=PROCESSING_SCHEMA,
        overwrite=False,
    )
    info(f"Wrote {len(scored_rows)} row(s) to DOCUMENTS_SCORED")

# Write REVIEW_QUEUE - all sources together
if review_rows:
    s.write_pandas(
        pd.DataFrame(review_rows),
        table_name='REVIEW_QUEUE',
        database=DB, schema=PROCESSING_SCHEMA,
        overwrite=False,
    )
    info(f"Wrote {len(review_rows)} row(s) to REVIEW_QUEUE "
         f"({len([r for r in review_rows if r['DOC_TYPE'] == 'unknown'])} unknown, "
         f"{len([r for r in review_rows if r.get('FLAG_REASONS', '').find('PIPELINE_ERROR') > -1])} errors, "
         f"{len([r for r in review_rows if r['DOC_TYPE'] not in (None, 'unknown')])} low score/missing)")


In [ ]:
# Update STATUS in DOCUMENTS_INGESTED

if auto_approve_ids:
    # Get parent DOC_IDs for auto-approved child docs
    id_list = ','.join(f"'{i}'" for i in auto_approve_ids)
    s.sql(f"""
        UPDATE {DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED
        SET STATUS = 'AUTO_APPROVED'
        WHERE DOC_ID IN (
            SELECT DOC_ID
            FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_CLASSIFIED
            WHERE CHILD_DOC_ID IN ({id_list})
        )
    """).collect()
    info(f"Updated {len(auto_approve_ids)} document(s) to AUTO_APPROVED")

review_child_ids = [
    r['CHILD_DOC_ID'] for r in review_rows
    if r['DOC_TYPE'] is not None   # exclude pipeline error docs
]
if review_child_ids:
    id_list = ','.join(f"'{i}'" for i in review_child_ids)
    s.sql(f"""
        UPDATE {DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED
        SET STATUS = 'HUMAN_REVIEW'
        WHERE DOC_ID IN (
            SELECT DOC_ID
            FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_CLASSIFIED
            WHERE CHILD_DOC_ID IN ({id_list})
        )
    """).collect()
    info(f"Updated {len(review_child_ids)} document(s) to HUMAN_REVIEW")

In [ ]:
# Summary
auto_count   = len(auto_approve_ids)
review_count = len(review_rows)
error_count  = len(error_docs)
total        = len(scored_rows) + error_count

print(f"\n Confidence gate summary ")
print(f"  Extraction model   : {EXTRACTION_MODEL}")
print(f"  Total processed    : {total}")
print(f"  Auto approved      : {auto_count} "
      f"({auto_count/total*100:.1f}% of total)" if total else "  Auto approved : 0")
print(f"  Human review       : {review_count} "
      f"({review_count/total*100:.1f}% of total)" if total else "  Human review  : 0")
print(f"  Pipeline errors    : {error_count}")

print(f"\n Score distribution ")
s.sql(f"""
    SELECT
        c.DOC_TYPE,
        sc.GATE_RESULT,
        COUNT(*)                            AS DOC_COUNT,
        ROUND(AVG(sc.COMPOSITE_SCORE), 3)  AS AVG_COMPOSITE,
        ROUND(MIN(sc.COMPOSITE_SCORE), 3)  AS MIN_COMPOSITE,
        ROUND(MAX(sc.COMPOSITE_SCORE), 3)  AS MAX_COMPOSITE,
        ROUND(AVG(sc.FIELD_COMPLETENESS), 3) AS AVG_COMPLETENESS,
        ROUND(AVG(sc.EXTRACT_CERTAINTY), 3)  AS AVG_CERTAINTY
    FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_SCORED sc
    JOIN {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_CLASSIFIED c
        ON sc.CHILD_DOC_ID = c.CHILD_DOC_ID
    WHERE sc.EXTRACTION_MODEL = '{EXTRACTION_MODEL}'
    GROUP BY c.DOC_TYPE, sc.GATE_RESULT
    ORDER BY c.DOC_TYPE, sc.GATE_RESULT
""").show()

print(f"\n Flag reason breakdown ")
s.sql(f"""
    SELECT
        f.value::VARCHAR        AS FLAG_REASON,
        COUNT(*)                AS OCCURRENCES
    FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_SCORED sc,
    LATERAL FLATTEN(input => PARSE_JSON(sc.FLAG_REASONS)) f
    WHERE sc.EXTRACTION_MODEL = '{EXTRACTION_MODEL}'
    GROUP BY f.value::VARCHAR
    ORDER BY OCCURRENCES DESC
""").show()

print(f"\n Review queue breakdown ")
s.sql(f"""
    SELECT
        DOC_TYPE,
        STATUS,
        COUNT(*) AS QUEUE_COUNT,
        f.value::VARCHAR AS FLAG_REASON
    FROM {DB}.{PROCESSING_SCHEMA}.REVIEW_QUEUE rq,
    LATERAL FLATTEN(input => PARSE_JSON(rq.FLAG_REASONS)) f
    GROUP BY DOC_TYPE, STATUS, f.value::VARCHAR
    ORDER BY QUEUE_COUNT DESC
""").show()